## Assigning risk to all narratives, Calculating response score and handling score

In [1]:
import polars as pl
import numpy as np
import pandas as pd
df = pl.read_csv("/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/CFPB_Labeled_Complaints_FilledOnly.csv")

texts = df["Narrative"].to_list()

y_severity = df["Severity"].to_numpy()
y_vulnerability = df["Vulnerability"].to_numpy()

In [2]:
from sklearn.model_selection import train_test_split

(
    X_train_text,
    X_test_text,
    ysev_train,
    ysev_test,
    yvul_train,
    yvul_test
) = train_test_split(
    texts,
    y_severity,
    y_vulnerability,
    test_size=0.2,
    random_state=42
)

In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=500000,
    ngram_range=(1,2),
    min_df=2,
    max_df=0.95,
    stop_words="english"
)

X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)

In [4]:
from sklearn.linear_model import Ridge

severity_model = Ridge(alpha=1.0)
severity_model.fit(X_train, ysev_train)

vulnerability_model = Ridge(alpha=1.0)
vulnerability_model.fit(X_train, yvul_train)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

In [5]:
sev_pred = severity_model.predict(X_test)
vul_pred = vulnerability_model.predict(X_test)

risk_pred = (
    0.7 * sev_pred +
    0.3 * vul_pred
)

risk_actual = (
    0.7 * ysev_test +
    0.3 * yvul_test
)

In [6]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Severity")
print("MAE:", mean_absolute_error(ysev_test, sev_pred))
print("RMSE:", np.sqrt(mean_squared_error(ysev_test, sev_pred)))
print("R2:", r2_score(ysev_test, sev_pred))

print("\nVulnerability")
print("MAE:", mean_absolute_error(yvul_test, vul_pred))
print("RMSE:", np.sqrt(mean_squared_error(yvul_test, vul_pred)))
print("R2:", r2_score(yvul_test, vul_pred))

Severity
MAE: 0.8332564725097971
RMSE: 1.042383804972557
R2: 0.5619795356496585

Vulnerability
MAE: 0.9650898665029621
RMSE: 1.2237614457585742
R2: 0.37148459577039694


In [7]:
data = pd.read_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/complaints_layer1.csv')

In [8]:
texts = (
    data["Narrative"]
    .to_list()
)

# Transform using trained TF-IDF vectorizer
X_full = tfidf.transform(texts)

# Predict severity and vulnerability
severity_pred = severity_model.predict(X_full)
vulnerability_pred = vulnerability_model.predict(X_full)



In [18]:
data['Risk'] = (
    0.7 * severity_pred +
    0.3 * vulnerability_pred
)

In [19]:
data['Risk']= np.clip(round(data['Risk'], 2), 0, 5)

In [20]:
data.sort_values(by='Risk', ascending=False)

,Date,Product,Issue,Narrative,Company,State,ZIP,Date sent,Response,Timely Response,Delay,Issue_Category,Risk
40194,2016-09-06,Lending,"Loan modification,collection,foreclosure","To whom it may concern, I have been trying to ...",PNC Bank N.A.,CA,910XX,2016-09-06,Closed with explanation,True,0,Loan Servicing & Repayment,5.00
285756,2016-11-25,Lending,"Loan servicing, payments, escrow account",I 'm a XXXX veteran and Homeowner who NEVER mi...,"CITIBANK, N.A.",AR,XXXXX,2016-11-25,Closed with explanation,True,0,Loan Servicing & Repayment,5.00
284200,2017-12-29,Lending,Struggling to pay mortgage,How many more people is this pirate mortgage c...,"Seterus, Inc.",NY,11375,2017-12-30,Closed with explanation,True,0,Loan Servicing & Repayment,4.91
234575,2015-08-25,Lending,"Loan modification,collection,foreclosure","Due to a series of financial misfortunes, my w...",JPMORGAN CHASE & CO.,NJ,07109,2015-08-25,Closed with explanation,True,0,Loan Servicing & Repayment,4.90
165581,2017-11-20,Lending,Struggling to pay mortgage,Seterus is very confused about their own lies ...,"Seterus, Inc.",NY,11375,2017-11-20,Closed with explanation,True,0,Loan Servicing & Repayment,4.81
...,...,...,...,...,...,...,...,...,...,...,...,...,...
142360,2017-11-02,Debt Collection,Attempts to collect debt not owed,I received a XXXX XXXX bill that was n't mine ...,NAVY FEDERAL CREDIT UNION,CA,93705,2017-11-02,Closed with explanation,True,0,Debt Collection,0.37
9713,2015-08-23,Payment & Money Services,Other transaction issues,never received first cardss : XXXX : XXXX XXXX,"WESTERN UNION COMPANY, THE",MO,647XX,2015-08-23,Closed with explanation,True,0,Other,0.37
256588,2018-01-23,Debt Collection,Attempts to collect debt not owed,i have never been to this school or received a...,"Ability Recovery Services, LLC",TX,776XX,2018-01-23,Closed with explanation,True,0,Debt Collection,0.37
26865,2017-12-30,Cards,Advertising,"Today, XXXX XXXX, 2017, I received a Netspend ...",Netspend Corporation,MI,48842,2017-12-30,Closed with explanation,True,0,Customer Service & Disclosures,0.35


In [21]:
data.to_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/complaints_layer2.csv')

In [24]:
data['Risk'].to_csv('/Users/abhaypai/Library/Mobile Documents/com~apple~CloudDocs/Job stuff/Pre 2026/Projects/Portfolio Projects/RBI/Data/risk_distribution.csv')